In [1]:
## calling and LLM from Langchain Ollama

from langchain_community.llms import Ollama

llm = Ollama(
    model="llama3.2",
    base_url="http://localhost:11434"
)


response = llm.invoke([{"role": "user", "content": "Hi, my name is Ashish"}])
print(response) 

/Users/ashishbansal/Documents/Training/Aistack_Morning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/qt/7qfpwbkx3cq0xb9m37811y000000gn/T/ipykernel_33583/3959044851.py:5: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


Hello Ashish! It's nice to meet you. How can I assist you today?


In [2]:
response = llm.invoke([{"role": "user", "content": "What is my name?"}])
print(response)

I don't have any information about you, so I'm not sure what your name is. Can you tell me your name?


In [3]:
from langgraph.graph import StateGraph, MessagesState
from langgraph.checkpoint.memory import InMemorySaver


checkpointer = InMemorySaver() # Short term memory

builder = StateGraph(MessagesState)

def chatbot(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}  # IMPORTANT: only return new message

builder.add_node("chatbot", chatbot)
builder.set_entry_point("chatbot")

graph = builder.compile(checkpointer=checkpointer)

In [4]:
config = {"configurable": {"thread_id": "user_2"}}

# First message
graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Ashish"}]},
    config
)

# Second message


{'messages': [HumanMessage(content='Hi, my name is Ashish', additional_kwargs={}, response_metadata={}, id='0ac5225b-df1c-42ab-9e93-6c20481999f4'),
  HumanMessage(content="Hello Ashish! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={}, id='1eb5c42c-9fb9-4dae-a9ba-5986a25494f3')]}

In [5]:
response = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config
)

print(response["messages"][-1].content)


Hello Ashish, it's nice to meet you too! To clarify, your name is indeed Ashish, and I'm happy to chat with you. We were just starting our conversation when you asked what your name was, but since we already established that, let's move on to something else. Would you like to talk about a particular topic or just have a casual conversation?


In [6]:
from langgraph.store.memory import InMemoryStore

from langchain.agents import create_agent


checkpointer = InMemorySaver() # Short term memory

store = InMemoryStore() # Long term memory

agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer,
    store=store
)

thread_config={"configurable": {"thread_id": "user_3"}}
agent.invoke({"messages": [{"role": "user", "content": "Hi, my name is Ashish"}]}, config=thread_config)
response=agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=thread_config)
print(response)


{'messages': [HumanMessage(content='Hi, my name is Ashish', additional_kwargs={}, response_metadata={}, id='8ec26147-43d4-4a50-a77a-1d46534aea4d'), HumanMessage(content="Hi Ashish! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={}, id='1509a1a4-d35f-4680-be6d-857de7087c59'), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='7ced1701-210b-45d4-9f8e-96185783c51c'), HumanMessage(content="Hello again, Ashish! You're wondering about your own name, aren't you? Don't worry, it happens to the best of us.\n\nTo clarify, our conversation just started, and we met earlier. You introduced yourself as Ashish, and I responded with a friendly greeting. That's why you mentioned asking if there was something I could help you with or chat, but then you asked about your own name out of curiosity!\n\nSo, to answer your question, your name is indeed Ashish!", additional_kwargs={}, respo

In [7]:
## LONG TERM MEMORY example 

namespace=("memories", "user_3") # Namespace for long term memory
memory_key="preferences" # Key to store the memory under

store.put(namespace, memory_key, {"name": "Ashish", "age": 30}) # Storing memory in long term store

saved=store.get(namespace, memory_key) # Retrieving memory from long term store
print(saved)

Item(namespace=['memories', 'user_3'], key='preferences', value={'name': 'Ashish', 'age': 30}, created_at='2026-07-17T02:50:45.786156+00:00', updated_at='2026-07-17T02:50:45.786157+00:00')


In [8]:
## chat bot example with long term memory and short term memory

checkpointer = InMemorySaver() # Short term memory
store = InMemoryStore() # Long term memory

agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer,
    store=store
)

def get_user_preferences(agent, thread_id):
    namespace=("users",thread_id)
    item=store.get(namespace, "preferences")
    if item:
        return item.value
    else:
        return {}

def save_user_preferences(agent, thread_id, preferences):
    namespace=("users",thread_id)
    store.put(namespace, "preferences", preferences)      

def chat(user_input, thread_id):
    preferences = get_user_preferences(agent, thread_id)
    
    if not preferences:
        # If we don't have preferences, ask the user and save them
        response = agent.invoke({"messages": [{"role": "user", "content": user_input}]}, config={"configurable": {"thread_id": thread_id}})
        print(response)
        
        # Extract preferences from response (this is just a placeholder, you'd need to implement actual extraction logic)
        extracted_preferences = {"name": "Ashish", "age": 30} 
        save_user_preferences(agent, thread_id, extracted_preferences)
    else:
        # If we have preferences, use them in the conversation
        response = agent.invoke({"messages": [{"role": "user", "content": f"{user_input} My name is {preferences['name']} and I am {preferences['age']} years old."}]}, config={"configurable": {"thread_id": thread_id}})
        print(response)

# Simulating a conversation
thread_id = "user_4"
chat("Hi, I am new here.My name is vipul", thread_id) # First message, preferences will be extracted and saved
chat("What is my name?", thread_id) # Second message, preferences will be used in the response




{'messages': [HumanMessage(content='Hi, I am new here.My name is vipul', additional_kwargs={}, response_metadata={}, id='1a155d94-fff5-4f0c-b5f8-6b34f1f069f1'), HumanMessage(content="Hello Vipul! Welcome to our community. It's great to meet you. How did you end up here, and what brings you to this place? Are you looking for something specific or just getting to know us?", additional_kwargs={}, response_metadata={}, id='7a302a3d-1f77-4186-899c-f182490ff307')]}
{'messages': [HumanMessage(content='Hi, I am new here.My name is vipul', additional_kwargs={}, response_metadata={}, id='1a155d94-fff5-4f0c-b5f8-6b34f1f069f1'), HumanMessage(content="Hello Vipul! Welcome to our community. It's great to meet you. How did you end up here, and what brings you to this place? Are you looking for something specific or just getting to know us?", additional_kwargs={}, response_metadata={}, id='7a302a3d-1f77-4186-899c-f182490ff307'), HumanMessage(content='What is my name? My name is Ashish and I am 30 year

In [9]:
thread_id = "user_5"
chat("Hello, I am Ashish. I like pizza.", thread_id) # First message, preferences will be extracted and saved
chat("What is I like?", thread_id) # Second

{'messages': [HumanMessage(content='Hello, I am Ashish. I like pizza.', additional_kwargs={}, response_metadata={}, id='389c69e0-cc4d-486b-ad00-cf5b33586fee'), HumanMessage(content="Hello Ashish! Nice to meet you. Pizza is a great choice - who doesn't love a good slice? What's your favorite kind of pizza - classic margherita, meat-lovers, or something more adventurous?", additional_kwargs={}, response_metadata={}, id='520c8191-3a7a-452f-8f00-e3298dd9072c')]}
{'messages': [HumanMessage(content='Hello, I am Ashish. I like pizza.', additional_kwargs={}, response_metadata={}, id='389c69e0-cc4d-486b-ad00-cf5b33586fee'), HumanMessage(content="Hello Ashish! Nice to meet you. Pizza is a great choice - who doesn't love a good slice? What's your favorite kind of pizza - classic margherita, meat-lovers, or something more adventurous?", additional_kwargs={}, response_metadata={}, id='520c8191-3a7a-452f-8f00-e3298dd9072c'), HumanMessage(content='What is I like? My name is Ashish and I am 30 years o